# Module 07 — Notebook 1: Virtual Environments

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain what a virtual environment is and the problem it solves
- Create a virtual environment with `uv venv` or `python -m venv`
- Activate and deactivate a venv from the terminal
- Verify which Python interpreter is active using `sys.executable`
- Inspect installed packages with `importlib.metadata`

**Estimated time:** ~20 minutes

## Why This Matters for AI Research Engineering

Every AI research project you work on will depend on specific versions of numpy, pandas, torch, or other packages. Without isolation, installing a new package for one project can break another. More critically: if you can't tell someone *exactly* which package versions produced your results, your experiment is not reproducible.

Virtual environments are the foundation that makes everything else in this module possible.

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains
import importlib.metadata
from pathlib import Path
print("Setup complete.")

## 1. The Problem: "It Works on My Machine"

If you've worked in Node.js, you already understand the solution: `node_modules/` isolates every project's dependencies. Without it, every `npm install` would pollute a global store and projects would break each other.

Python historically worked exactly that way — everything went into one global site-packages directory. Virtual environments fix this.

```
Without a venv (global Python):
  /usr/local/lib/python3.11/site-packages/
    numpy 1.23     ← which project does this belong to?
    pandas 1.5

With venvs (isolated):
  project-a/.venv/lib/python3.11/site-packages/
    numpy 1.26     ← only for project-a

  project-b/.venv/lib/python3.11/site-packages/
    numpy 1.20     ← only for project-b, no conflict
```

This is the same reason AI research codebases pin exact versions — so that "install and run" actually works.

## 2. Creating a Virtual Environment

You run these commands in your **terminal**, not in a notebook cell. The commands create a `.venv/` directory containing an isolated Python interpreter.

```bash
# Modern approach — uv is much faster than pip:
uv venv .venv

# Classic approach — built into Python, always available:
python -m venv .venv
```

Both commands create the same structure:

```
.venv/
  bin/
    python       ← the isolated interpreter
    pip          ← the isolated pip
  lib/
    python3.11/
      site-packages/   ← installed packages live here
  pyvenv.cfg   ← metadata about the venv
```

**Convention:** name it `.venv` (with a leading dot) so it's hidden from directory listings and recognized automatically by VS Code and other tools.

## 3. Activating and Deactivating

Creating a venv doesn't automatically use it — you need to **activate** it first. Activation tells your shell to prefer the venv's Python over the system Python.

```bash
# Mac / Linux — activate:
source .venv/bin/activate

# Windows (Command Prompt) — activate:
.venv\Scripts\activate.bat

# Windows (PowerShell) — activate:
.venv\Scripts\Activate.ps1

# Any platform — deactivate when done:
deactivate
```

After activation, your shell prompt shows `(.venv)` and `which python` points to `.venv/bin/python`.

```bash
# Verify you're in the right environment:
which python          # Mac/Linux → /your/project/.venv/bin/python
where python          # Windows  → C:\...\project\.venv\Scripts\python.exe
python --version      # shows the Python version inside the venv
```

> **VS Code shortcut:** If your project root contains a `.venv/` folder, VS Code detects it automatically and activates it in the integrated terminal.

## 4. Inspecting the Active Interpreter from Python

Once you're inside a Python session (or a Jupyter kernel), you can check which interpreter is running using `sys`.

In [ ]:
import sys
from pathlib import Path

interpreter = Path(sys.executable)
version_info = sys.version_info

print(f"Python interpreter: {interpreter}")
print(f"Version: {version_info.major}.{version_info.minor}.{version_info.micro}")
print(f"sys.path[:3]: {sys.path[:3]}")

## 5. Inspecting Installed Packages with `importlib.metadata`

The standard-library module `importlib.metadata` lets you query your environment programmatically — no shell commands needed.

In [ ]:
import importlib.metadata

# List all installed distributions
all_packages = {d.name: d.version for d in importlib.metadata.distributions()}
print(f"Total installed packages: {len(all_packages)}")

# Get a specific package version
numpy_version = importlib.metadata.version("numpy")
print(f"numpy version: {numpy_version}")

## Exercise 1 — Inspect the Python Version

Using `sys.version_info`, extract the major and minor version numbers and store them in `major` and `minor`.

In [ ]:
import sys

# YOUR CODE HERE
# Hint: sys.version_info.major and sys.version_info.minor
major = None  # int
minor = None  # int

In [ ]:
check_type(major, int, "major is an int")
check_type(minor, int, "minor is an int")
check_equal(major, 3, "major version is 3")
check_equal(minor >= 9, True, "minor version is 9 or higher")

## Exercise 2 — Get the pip Version

Use `importlib.metadata.version()` to get the installed version of `pip` as a string and store it in `pip_version`.

In [ ]:
import importlib.metadata

# YOUR CODE HERE
# Hint: importlib.metadata.version("pip") returns a string like "24.0"
pip_version = None  # str

In [ ]:
check_type(pip_version, str, "pip_version is a string")
check_equal("." in pip_version, True, "pip_version looks like a version string (contains '.')")

## Exercise 3 — Check if a Package is Installed

Write code that sets `numpy_installed` to `True` if numpy is installed, `False` if it's not. Use a `try/except` block with `importlib.metadata.PackageNotFoundError`.

In [ ]:
import importlib.metadata

# YOUR CODE HERE
# Hint: try importlib.metadata.version("numpy") — if it raises PackageNotFoundError, it's not installed
numpy_installed = None  # bool

In [ ]:
check_type(numpy_installed, bool, "numpy_installed is a bool")
check_equal(numpy_installed, True, "numpy is installed in this environment")

## Wrap-Up

| Command / Code | What it does |
|---|---|
| `uv venv .venv` | Create a venv using uv (fast) |
| `python -m venv .venv` | Create a venv using stdlib |
| `source .venv/bin/activate` | Activate on Mac/Linux |
| `.venv\Scripts\activate` | Activate on Windows |
| `deactivate` | Return to system Python |
| `sys.executable` | Path to the active Python interpreter |
| `sys.version_info` | Structured version info (major, minor, micro) |
| `importlib.metadata.version("pkg")` | Get installed version of a package |
| `importlib.metadata.distributions()` | List all installed packages |

**Next:** Notebook 2 — Package Management (installing, pinning, and freezing dependencies)